# Tutorial 7b: Asking questions of a coupled model (with PyMC)

Estimated time: 25-35 minutes. Optional, and best read straight after Tutorial 7.

## What you need installed

| Need | Why | Install |
|---|---|---|
| `bayesian-metamodeling` + `[tutorials]` | framework, numpy, matplotlib | `pip install -e ".[tutorials]"` |
| **PyMC** | this notebook's whole point: PyMC does the sampling | `conda env create -f environment-all.yml` (env `py312_bayesmm_all`) |

T7 needed no backend at all. This one does, and the difference between those two sentences
is the subject of the notebook.

## What you'll be able to do afterwards

- **Ask a conditional question** of a coupled metamodel — "I measured this; what does it
  imply about that?" — which T7's machinery could not express.
- **Run inference backwards** through a chain of models: observe a downstream output,
  recover an upstream input you never measured.
- **Read convergence diagnostics** (r-hat, divergences, ESS) and say when a result is not
  yet trustworthy.
- **Say when the fast path does not apply**, and what happens then.

## How you'll know you got it

Given a two-model chain and a measurement of the last variable, you can predict — before
running anything — whether the *first* variable will move, and say which sampling method
makes that possible.


## Where this sits

T7 established what a coupling is: a factor multiplied into a product of priors, reshaping
a round cloud into a ridge. It sampled that joint with a **random walk**, and said plainly
why — a fitted surrogate's `log_prob` is a black box with no gradient, so a gradient-free
sampler is what the model admits.

That is true in general. It is not true for `pymc_gp`.

| | T7 | T7b (here) |
|---|---|---|
| sampler | random-walk Metropolis | NUTS, via PyMC |
| surrogate treated as | a black box you call | a formula you can differentiate |
| works with | any backend | `pymc_gp` only (falls back otherwise) |
| diagnostics | acceptance rate | r-hat, divergences, ESS |
| can condition on a measurement | yes, but inefficiently | yes |
| needs PyMC installed | no | yes |

Both sample **the same distribution**. Step 1 checks that rather than asserting it — if two
implementations of one density disagree, both keep producing plausible numbers, so this is
worth a cell.


## Why there can be a faster path at all

Recall from T5: despite its name, `pymc_gp` is **Bayesian linear regression**. Fitting it
leaves a posterior over three things — the slope(s) `W`, the intercept, and a noise scale
`sigma` — stored as several hundred draws in the artifact.

That means the surrogate's predictive density is not an opaque function. It is

$$p(y \mid x) \;=\; \frac{1}{S}\sum_{s=1}^{S} \mathrm{Normal}\!\big(y;\; xW_s + b_s,\; \sigma_s\big)$$

an average of `S` Gaussians — one per posterior draw. Every operation in it is
differentiable, so it can be written as a symbolic expression and handed to a gradient
sampler.

Two things follow, and the second is easy to miss:

1. **Gradients.** NUTS proposes along the geometry of the distribution instead of blindly.
2. **The surrogate's own uncertainty comes along.** Calling `log_prob` as a black box gives
   one number; writing the mixture symbolically keeps all `S` draws in play, so uncertainty
   about *the surrogate's parameters* propagates into the metamodel. That is the reason to
   have fitted a Bayesian surrogate rather than a curve.

`sbi_npe` is a neural density estimator in PyTorch. Its gradients exist, but reaching them
from PyMC's expression system needs an adapter nobody has written, so it takes the T7 path.
Step 5 shows what that looks like.


In [ ]:
# Cross-platform setup — same opening cell as every other tutorial.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap  # noqa: E402

root = bootstrap()
ROOT = root
print("Repo root:", root)

try:
    import pymc  # noqa: F401
    PYMC_AVAILABLE = True
    print("PyMC found — every step below will run.")
except ImportError:
    PYMC_AVAILABLE = False
    print("=" * 72)
    print("PREFLIGHT: PyMC is missing, so the sampling steps will skip.")
    print("  conda env create -f environment-all.yml")
    print("  conda activate py312_bayesmm_all")
    print("The explanations still read fine; only the cells that sample need it.")
    print("=" * 72)


## Step 1: two samplers, one distribution

Before trusting a faster method, check it agrees with the slower one you already believe.

The model is T7's two-variable case, chosen because its answer is available **on paper**:
normal priors and a linear-Gaussian coupling give a Gaussian joint whose mean and standard
deviation come from inverting a 2x2 matrix. So there are three numbers to compare, not two,
and the closed form is the referee.


In [ ]:
import numpy as np

if not PYMC_AVAILABLE:
    print("Step 1 SKIPPED — needs PyMC (see preflight above).")
    nuts_s = rw_s = None
else:
    from bayesian_metamodeling.meta.compiler import compile_metamodel
    from bayesian_metamodeling.meta.ir import (
        CouplingFactorIR, MetamodelIR, PriorFactorIR, VariableIR,
    )
    from bayesian_metamodeling.meta.joint_sampling import sample_joint
    from bayesian_metamodeling.meta.nuts_sampling import sample_nuts

    mx, sx, my, sy, alpha, beta, sigma = 1.0, 0.5, 0.0, 2.0, 1.5, -0.4, 0.3
    pair = MetamodelIR(
        name="pair",
        variables=[VariableIR(name="x"), VariableIR(name="y")],
        factors=[
            PriorFactorIR(variable="x", distribution={"kind": "normal", "loc": mx, "scale": sx}),
            PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": my, "scale": sy}),
            CouplingFactorIR(coupling_type="gaussian_link", source="x", target="y",
                             transform={"kind": "affine", "alpha": alpha, "beta": beta},
                             sigma=sigma),
        ],
    )

    lam = np.array([[1/sx**2 + alpha**2/sigma**2, -alpha/sigma**2],
                    [-alpha/sigma**2, 1/sy**2 + 1/sigma**2]])
    h = np.array([mx/sx**2 - alpha*beta/sigma**2, my/sy**2 + beta/sigma**2])
    cov = np.linalg.inv(lam)
    exact_mean, exact_sd = cov @ h, np.sqrt(np.diag(cov))

    nuts_s, nuts_d = sample_nuts(compile_metamodel(pair), draws=3000, tune=1000,
                                 chains=2, seed=11)
    rw_s, rw_d = sample_joint(compile_metamodel(pair), draws=9000, tune=2500,
                              chains=2, seed=11)

    print(f"{'':<22}{'mean x':>9}{'mean y':>9}{'sd x':>9}{'sd y':>9}")
    print(f"{'closed form (truth)':<22}{exact_mean[0]:>9.3f}{exact_mean[1]:>9.3f}"
          f"{exact_sd[0]:>9.3f}{exact_sd[1]:>9.3f}")
    for label, s in (("NUTS (this notebook)", nuts_s), ("random walk (T7)", rw_s)):
        print(f"{label:<22}{s['x'].mean():>9.3f}{s['y'].mean():>9.3f}"
              f"{s['x'].std():>9.3f}{s['y'].std():>9.3f}")


**All three rows agree.** That is the licence to use the faster one: it is not a
different model or an approximation, it is the same density explored more cleverly.

Keep the habit. Any time a pipeline offers you a faster path, the first question is whether
it computes the same thing — and a problem with a known answer is the cheapest way to ask.


## Step 2: what the gradients actually buy

Both runs above landed on the right answer, so why bother? Because "draws" and
"information" are not the same thing.

A random walk proposes a step in a random direction. In a distribution shaped like a narrow
ridge — which is what a coupling *makes* — most directions point off the ridge and get
rejected, so consecutive draws are highly correlated. **Effective sample size** counts how
many independent draws your correlated ones are worth.


In [ ]:
if not PYMC_AVAILABLE or nuts_s is None:
    print("Step 2 SKIPPED — needs Step 1.")
else:
    n_nuts = nuts_s["x"].size
    n_rw = rw_s["x"].size
    print(f"{'':<20}{'draws':>8}{'ESS(x)':>9}{'per draw':>10}")
    print(f"{'NUTS':<20}{n_nuts:>8}{nuts_d['ess']['x']:>9.0f}"
          f"{nuts_d['ess']['x']/n_nuts:>9.0%}")
    print(f"{'random walk':<20}{n_rw:>8}{rw_d['ess']['x']:>9.0f}"
          f"{rw_d['ess']['x']/n_rw:>9.0%}")
    print()
    print("Diagnostics NUTS reports that a random walk cannot:")
    print(f"  r-hat (max over variables) = {max(nuts_d['r_hat'].values()):.4f}   "
          "(agreement between independent chains; >1.01 is a warning)")
    print(f"  divergences                = {nuts_d['divergences']}   "
          "(geometry the sampler could not follow; >0 means distrust the result)")


**Read `per draw`.** Both samplers are correct; one gets far more information from each
draw. On this small problem it hardly matters. On the real four-surrogate metamodel in
`projects/tcr_signaling` the same comparison is 0.7% against 58%, and three variables came
back from the random walk with an effective sample size in the *tens* — summaries that
looked fine and meant nothing.

**And note what r-hat and divergences are for.** They are not scores; they are alarms.
`r-hat` compares independent chains — if they explored different regions, they disagree and
r-hat rises above 1. A divergence means the sampler hit geometry it could not integrate
through. Either one says: do not report these numbers yet.

> **r-hat needs at least two chains.** With `chains=1` it is undefined, and the framework
> reports `None` rather than a number — a NaN would quietly satisfy any `r_hat < 1.01`
> check and make a single-chain run look flawless.


## Step 3: the question T7 could not ask

Here is the gap. T7 could draw the joint distribution of a coupled model. It had no way to
say **"I measured this one"** — and that is the question people actually bring to a
metamodel.

The spec now takes an `observed` block:

```json
{"observed": {"y": 3.0}}
```

An observed variable is **clamped**: not drawn, not proposed, held exactly at its value
while every factor that mentions it is evaluated there. It leaves the sample space, so the
sampler works in one fewer dimension.

For the two-variable model, conditioning also has an answer on paper, so once again we can
check rather than trust.


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 3 SKIPPED — needs PyMC.")
else:
    y_measured = 3.0
    conditioned = MetamodelIR(
        name="pair_conditioned",
        variables=[VariableIR(name="x"), VariableIR(name="y")],
        factors=list(pair.factors),
        observed={"y": y_measured},
    )
    cs, cd = sample_nuts(compile_metamodel(conditioned), draws=3000, tune=1000,
                         chains=2, seed=3)

    # p(x | y) for normal prior + linear-Gaussian coupling, by completing the square.
    prec = 1/sx**2 + alpha**2/sigma**2
    post_mean = (mx/sx**2 + alpha*(y_measured - beta)/sigma**2) / prec
    post_sd = 1/np.sqrt(prec)

    print(f"before measuring:  x = {nuts_s['x'].mean():+.3f} +- {nuts_s['x'].std():.3f}")
    print(f"after y = {y_measured}:     x = {cs['x'].mean():+.3f} +- {cs['x'].std():.3f}")
    print(f"closed form:       x = {post_mean:+.3f} +- {post_sd:.3f}")
    print()
    print(f"y is held exactly: all draws == {y_measured}? "
          f"{bool(np.all(cs['y'] == y_measured))}")
    print(f"variables actually sampled: {cd['free_variables']}   (y is not among them)")


**Two things happened.** `x` moved — the measurement of `y` told us something about it —
and `x` got *narrower*, because a measurement is information. The closed form confirms both.

This is inference, in the ordinary sense: a belief updated by evidence. And note it is only
possible because the coupling was written as a factor rather than as a generative step. A
factor is symmetric, so evidence can enter at either end.


## Step 4: backwards, through two models

Now the payoff, and the reason this matters for real work.

Chain two fitted surrogates:

- **Model A** learned `y` from `x` (truth: `y = 2x + 1`)
- **Model B** learned `z` from `y` (truth: `z = 0.5y - 0.3`)

Then measure **`z`** — the very last quantity — and ask about **`x`**, which is two models
upstream and was never measured.

Under `--method propagate` this question is not merely inefficient, it is *unanswerable*:
propagation only ever pushes values downstream. Predict before you run: if `z = 2.2`, what
should `x` be? (The algebra is in the cell's output, so commit to a number first.)


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 4 SKIPPED — needs PyMC.")
    chain_free = chain_cond = None
else:
    from bayesian_metamodeling.meta.ir import SurrogateLikelihoodFactorIR
    from bayesian_metamodeling.surrogates.backends import fit_backend_model

    rng = np.random.default_rng(0)
    # Model A: fit y = 2x + 1 from 100 noisy runs.
    xs = rng.uniform(-2, 3, 100)
    A = fit_backend_model(
        backend="pymc_gp", x=xs.reshape(-1, 1),
        y=(2*xs + 1 + rng.normal(0, 0.25, 100)).reshape(-1, 1),
        input_names=["x"], output_names=["y"],
        backend_config={"draws": 400, "tune": 400, "chains": 1, "target_accept": 0.9}, seed=0)
    # Model B: fit z = 0.5y - 0.3, independently.
    ys = rng.uniform(-3, 7, 100)
    B = fit_backend_model(
        backend="pymc_gp", x=ys.reshape(-1, 1),
        y=(0.5*ys - 0.3 + rng.normal(0, 0.2, 100)).reshape(-1, 1),
        input_names=["y"], output_names=["z"],
        backend_config={"draws": 400, "tune": 400, "chains": 1, "target_accept": 0.9}, seed=1)

    def chain_ir(observed):
        return MetamodelIR(
            name="chain", variables=[VariableIR(name=n) for n in ("x", "y", "z")],
            factors=[
                PriorFactorIR(variable="x", distribution={"kind": "normal", "loc": 0.0, "scale": 2.0}),
                PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": 0.0, "scale": 6.0}),
                PriorFactorIR(variable="z", distribution={"kind": "normal", "loc": 0.0, "scale": 4.0}),
                SurrogateLikelihoodFactorIR(surrogate_ref="A", inputs=["x"], outputs=["y"]),
                SurrogateLikelihoodFactorIR(surrogate_ref="B", inputs=["y"], outputs=["z"]),
            ],
            observed=observed)

    surr = {"A": A, "B": B}
    chain_free, _ = sample_nuts(compile_metamodel(chain_ir({})), draws=2000, tune=1000,
                                chains=2, seed=7, surrogates=surr)
    z_measured = 2.2
    chain_cond, chain_d = sample_nuts(compile_metamodel(chain_ir({"z": z_measured})),
                                      draws=2000, tune=1000, chains=2, seed=7, surrogates=surr)

    print(f"{'':<26}{'x':>18}{'y':>18}{'z':>10}")
    print(f"{'before measuring':<26}"
          f"{chain_free['x'].mean():>10.3f} +-{chain_free['x'].std():<6.3f}"
          f"{chain_free['y'].mean():>10.3f} +-{chain_free['y'].std():<6.3f}"
          f"{chain_free['z'].mean():>10.3f}")
    print(f"{'after measuring z = 2.2':<26}"
          f"{chain_cond['x'].mean():>10.3f} +-{chain_cond['x'].std():<6.3f}"
          f"{chain_cond['y'].mean():>10.3f} +-{chain_cond['y'].std():<6.3f}"
          f"{chain_cond['z'].mean():>10.3f}")
    y_true = (z_measured + 0.3) / 0.5
    print(f"\nalgebra, if the fits were exact:  y = (z+0.3)/0.5 = {y_true:.2f}, "
          f"x = (y-1)/2 = {(y_true - 1)/2:.2f}")
    print(f"r-hat={max(chain_d['r_hat'].values()):.4f}  divergences={chain_d['divergences']}  "
          f"worst ESS={min(chain_d['ess'].values()):.0f}")


**Read the `x` column.** Before the measurement it sat near its prior, wide. After
measuring `z` — a quantity two models away, with `x` appearing in neither the measurement
nor Model B — it lands within a few hundredths of the algebraic answer, and roughly seven
times narrower.

Nothing about this is special-cased. `x` and `z` are connected through a chain of factors,
and conditioning propagates along the whole chain in both directions. **That is what having
a joint model buys you**, and it is the thing the propagate path structurally cannot do.

This generalises to the question worth remembering: *any* variable can be conditioned on
*any* other, as long as a path of factors connects them. Measure an output, learn about an
input. Measure one model's variable, learn about another model's. You are not restricted to
the direction the simulators happen to run in — and that, rather than speed, is the real
argument for building the joint at all.


## Step 5: when the fast path does not apply

`--method nuts` needs every surrogate to have a symbolic density. An `sbi_npe` surrogate is
a neural network, and the framework will not pretend otherwise: it says so and falls back
to the T7 sampler, which handles black boxes by design.

The refusal is deliberately loud. A performance flag that silently gives you the slow path
is worse than no flag, because you would report the fast one in your methods section.


In [ ]:
from bayesian_metamodeling.meta.nuts_sampling import nuts_supported

if not PYMC_AVAILABLE:
    print("Step 5 SKIPPED — needs PyMC.")
else:
    ok, reason = nuts_supported(chain_ir({}), surr)
    print(f"two pymc_gp surrogates -> NUTS supported? {ok}")

    class _NeuralSurrogate:
        """Stands in for a fitted sbi_npe model: callable, but not differentiable here."""
        def log_prob(self, inputs, outputs):
            raise NotImplementedError

    ok2, reason2 = nuts_supported(chain_ir({}), {"A": A, "B": _NeuralSurrogate()})
    print(f"one of them swapped for a neural one -> supported? {ok2}")
    print(f"  reason given: {reason2}")
    print()
    print("From the CLI this prints '--method nuts unavailable (...); falling back to")
    print("joint.' and then runs — you get an answer, and you are told how.")


## Recap

- **`pymc_gp` is linear regression**, so its predictive density is a mixture of Gaussians
  you can differentiate. That is why a gradient sampler is possible here at all, and why it
  is not possible for a neural surrogate.
- **NUTS and the random walk sample the same distribution.** Step 1 checked that against a
  closed form. The faster one is not an approximation.
- **Gradients buy information per draw, and diagnostics.** r-hat and divergences are alarms
  that a random walk cannot raise.
- **`observed` is what makes a metamodel answerable.** Clamp what you measured; everything
  connected updates.
- **Conditioning runs in every direction.** Measure a downstream output, recover an upstream
  input. This is the capability that justifies building a joint model rather than a
  pipeline of one-way predictions.
- **The fast path refuses out loud** when it does not apply.

**Where next:** T8 chains three couplings and budgets the noise along them. The real
four-surrogate metamodel is `projects/tcr_signaling/notebooks/03_metamodel_inference.ipynb`
— the same commands, on models fitted to published data.


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: PyMC is missing` | no PyMC in this kernel | `conda env create -f environment-all.yml`, then use the `py312_bayesmm_all` kernel |
| `--method nuts unavailable (... not a diagonal pymc_gp ...)` | one surrogate is `sbi_npe`, or was fitted with `output_correlation: "full"` | Expected. It falls back to `joint` and still answers; refit with `pymc_gp` if you want the fast path |
| `r_hat` is `None` | you used one chain, where r-hat is undefined | pass `--chains 2` or more; the framework refuses to report a NaN that would pass every check |
| `divergences > 0` | the sampler hit geometry it could not integrate | raise `target_accept`, or look for a coupling `sigma` far tighter than the priors it constrains |
| conditioning barely moves a variable | no path of factors connects it to what you measured | check the built graph — an isolated variable stays at its prior, correctly |


## Final check

Asserts what this notebook claims, each in a way that could fail on a run that executed but
demonstrated nothing: NUTS agrees with the closed form; conditioning both moves and narrows
the conditioned-on variable; the observed variable is held exactly; inference reaches
backwards through two models to within 0.15 of the algebraic answer; and a non-linear
surrogate is refused rather than silently mis-sampled.


In [ ]:
if not PYMC_AVAILABLE:
    print("\n[T7b self-check OK] every sampling step skipped per preflight (PyMC absent).")
else:
    # 1. NUTS matched the closed form (Step 1).
    assert abs(nuts_s["x"].mean() - exact_mean[0]) < 0.05, "NUTS mean disagrees with closed form"
    assert abs(nuts_s["x"].std() - exact_sd[0]) < 0.05 * exact_sd[0], "NUTS sd disagrees"

    # 2. The two samplers agree — the guard against two implementations drifting apart.
    assert abs(nuts_s["x"].mean() - rw_s["x"].mean()) < 0.08, (
        "NUTS and the random walk no longer encode the same density"
    )

    # 3. Conditioning moved AND narrowed x, and held y exactly (Step 3).
    assert np.all(cs["y"] == y_measured), "observed variable was sampled instead of clamped"
    assert cs["x"].mean() > nuts_s["x"].mean() + 0.15, "conditioning did not move x"
    assert cs["x"].std() < nuts_s["x"].std(), "a measurement must not widen what it informs"

    # 4. Inference ran backwards through two models (Step 4) — the notebook's headline.
    _x_expected = ((z_measured + 0.3) / 0.5 - 1) / 2
    _err = abs(chain_cond["x"].mean() - _x_expected)
    assert _err < 0.15, (
        f"x recovered as {chain_cond['x'].mean():.3f}, algebra says {_x_expected:.3f} "
        f"(off by {_err:.3f}) — information is not reaching upstream"
    )
    assert chain_cond["x"].std() < 0.5 * chain_free["x"].std(), (
        "measuring z should sharply narrow x; it did not"
    )
    assert chain_d["divergences"] == 0, f"{chain_d['divergences']} divergences — distrust this"

    # 5. A non-differentiable surrogate is refused, not mis-sampled (Step 5).
    assert not ok2 and "pymc_gp" in reason2, "a neural surrogate must not be claimed as NUTS-able"

    print(f"\n[T7b self-check OK] NUTS==closed form; conditioning moved x "
          f"{nuts_s['x'].mean():+.2f} -> {cs['x'].mean():+.2f}; backwards inference "
          f"x={chain_cond['x'].mean():.3f} vs algebra {_x_expected:.3f}; "
          f"r-hat={max(chain_d['r_hat'].values()):.4f}, {chain_d['divergences']} divergences.")
